# Structured Output from LLMs: Constrained JSON Decoding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/agents/structured_output_constrained_decoding.ipynb)

Companion notebook to [the blog post](https://sesen.ai/blog/structured-output-llm-constrained-decoding).

There are three ways to get valid JSON out of a language model. Ask nicely and
retry; validate and repair; or make invalid tokens unsamplable. Only the third
cannot fail, and we build it here from scratch on GPT-2 small.

Contents:

1. The task, and how often plain prompting produces valid JSON
2. The schema as a character automaton
3. Lifting the automaton onto GPT-2's 50,257-token vocabulary
4. One decoding loop, four strategies, measured side by side
5. What the mask costs: sub-word misalignment and the grammar's spelling
6. A scripted model, for running all of this with no download

Everything is CPU-only inference. No training, no API key.

In [ ]:
!pip install -q transformers torch

import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, "
      f"{len(tokenizer):,} tokens in the vocabulary")

## 1. The task

Turn a sentence into a record. Three keys, three types: a string, an integer and a
boolean. Three worked examples go in the prompt, which is as much help as a
124M-parameter model is going to get.

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Record:
    sentence: str
    name: str
    age: int
    verified: bool

    @property
    def target(self):
        flag = "true" if self.verified else "false"
        return f'{{"name": "{self.name}", "age": {self.age}, "verified": {flag}}}'


SHOTS = [
    Record("Grace Hopper is 45 and her account is verified.", "Grace Hopper", 45, True),
    Record("Alan Turing, 41, has not verified his account.", "Alan Turing", 41, False),
    Record("Katherine Johnson is 52 with a verified account.", "Katherine Johnson", 52, True),
]

EVAL = [
    Record("Ada Lovelace is 36 and her account is verified.", "Ada Lovelace", 36, True),
    Record("Claude Shannon, 32, has not verified his account.", "Claude Shannon", 32, False),
    Record("Barbara Liskov is 48 with a verified account.", "Barbara Liskov", 48, True),
    Record("John McCarthy is 29 and has not verified.", "John McCarthy", 29, False),
    Record("Radia Perlman, 55, has a verified account.", "Radia Perlman", 55, True),
    Record("Geoffrey Hinton is 61 and is not verified.", "Geoffrey Hinton", 61, False),
    Record("Fei Fei Li is 39 with a verified account.", "Fei Fei Li", 39, True),
    Record("Yann LeCun, 44, has not verified his account.", "Yann LeCun", 44, False),
]

INSTRUCTION = "Turn each sentence into a JSON record with the keys name, age and verified.\n\n"


def build_prompt(record):
    parts = [INSTRUCTION]
    for shot in SHOTS:
        parts.append(f"Sentence: {shot.sentence}\nJSON: {shot.target}\n\n")
    parts.append(f"Sentence: {record.sentence}\nJSON: ")
    return "".join(parts)


print(build_prompt(EVAL[0]))

Before writing any masking code, look at what the tokeniser does to a target record.
This is the whole difficulty in one line of output.

In [ ]:
target = EVAL[0].target
print(target)
print([tokenizer.decode([i]) for i in tokenizer(target).input_ids])

`{"` is a single token. The opening brace and the first quote of the first key
belong to two different places in the schema, and the tokeniser has glued them
together. A mask applied character by character would be too late.

## 2. The schema as a character automaton

One schema with fixed keys is a chain of fixed text with holes in it. Blue segments
in the post's diagram match exact text; the holes are a bounded string, one to three
digits, and `true` or `false`. Bounding the holes keeps the state count finite, so
every path terminates.

In [ ]:
MAX_NAME_CHARS, MAX_AGE_DIGITS = 24, 3
STRING_CHARS = frozenset(
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .'-"
)
FIELD_KINDS = {
    "name": ("str", MAX_NAME_CHARS),
    "age": ("int", MAX_AGE_DIGITS),
    "verified": ("bool", None),
}
ACCEPT = ("accept", 0, 0)
_BOOL_WORDS = {1: "true", 2: "false"}


def build_segments(order=("name", "age", "verified"), space_after_colon=True):
    """Serialise the schema into literal text with holes where the values go."""
    colon = ": " if space_after_colon else ":"
    segments = []
    for i, key in enumerate(order):
        opener = "{" if i == 0 else ", "
        kind, param = FIELD_KINDS[key]
        quote = '"' if kind == "str" else ""
        segments.append(("lit", f'{opener}"{key}"{colon}{quote}'))
        segments.append((kind, param))
    segments.append(("lit", "}"))
    return segments


class Grammar:
    """A character DFA that accepts exactly the strings matching the schema.

    A state is (segment index, characters matched in that segment, branch). The
    branch field is only used by the boolean, which is an alternation between two
    literals and has to remember which one it committed to.
    """

    def __init__(self, segments=None):
        self.segments = segments or build_segments()

    def start(self):
        return self._enter(0)

    def _enter(self, seg_idx):
        return ACCEPT if seg_idx >= len(self.segments) else (seg_idx, 0, 0)

    def advance(self, state, ch):
        """Consume one character. Returns the next state, or None if ch is illegal."""
        if state == ACCEPT:
            return None
        seg_idx, pos, branch = state
        kind, param = self.segments[seg_idx]

        if kind == "lit":
            text = str(param)
            if ch != text[pos]:
                return None
            pos += 1
            return self._enter(seg_idx + 1) if pos == len(text) else (seg_idx, pos, 0)

        if kind == "str":
            if ch == '"':                      # the string owns its closing quote
                return self._enter(seg_idx + 1)
            if pos >= int(param) or ch not in STRING_CHARS:
                return None
            return (seg_idx, pos + 1, 0)

        if kind == "int":
            if ch.isdigit():
                if pos >= int(param):
                    return None
                # JSON forbids leading zeros, so a leading 0 is a complete number.
                return (seg_idx, int(param) if (pos == 0 and ch == "0") else pos + 1, 0)
            return self.advance(self._enter(seg_idx + 1), ch) if pos > 0 else None

        if kind == "bool":
            if pos == 0:
                branch = {"t": 1, "f": 2}.get(ch, 0)
                return None if branch == 0 else (seg_idx, 1, branch)
            word = _BOOL_WORDS[branch]
            if ch != word[pos]:
                return None
            pos += 1
            return self._enter(seg_idx + 1) if pos == len(word) else (seg_idx, pos, branch)

        raise ValueError(kind)

    def consume(self, state, text):
        for ch in text:
            state = self.advance(state, ch)
            if state is None:
                return None
        return state

    def accepts(self, text):
        return self.consume(self.start(), text) == ACCEPT


grammar = Grammar()
print("target accepted:", grammar.accepts(EVAL[0].target))
for bad in ['{"name": "Ada", "age": 36}',
            '{"name": "Ada", "age": 36, "verified": yes}',
            '{"name": "Ada", "age": 036, "verified": true}']:
    print(f"  rejected: {grammar.accepts(bad) is False}  {bad}")

### Try it

Change `MAX_NAME_CHARS` to 4 and re-run. The grammar still accepts only valid JSON,
but no name longer than four characters can be written, so the model is forced to
truncate. A grammar is a hard constraint in both directions.

## 3. Lifting the automaton onto the vocabulary

A token is legal in a state when every one of its characters is legal in turn. The
answer depends only on the state, so cache it: a full record visits around 40
distinct states and reuses each mask many times.

In [ ]:
def bytes_to_unicode():
    """GPT-2's byte-to-printable-character map (Radford et al., 2019)."""
    printable = (list(range(ord("!"), ord("~") + 1))
                 + list(range(ord("\xa1"), ord("\xac") + 1))
                 + list(range(ord("\xae"), ord("\xff") + 1)))
    mapped, shift = list(printable), 0
    for byte in range(256):
        if byte not in printable:
            printable.append(byte)
            mapped.append(256 + shift)
            shift += 1
    return dict(zip(printable, (chr(c) for c in mapped)))


def vocab_strings(tokenizer):
    """Decode every vocabulary entry to the text it contributes, spaces included."""
    byte_decoder = {v: k for k, v in bytes_to_unicode().items()}
    out = []
    for token in tokenizer.convert_ids_to_tokens(range(len(tokenizer))):
        try:
            raw = bytearray(byte_decoder[c] for c in token)
        except KeyError:                       # special tokens such as <|endoftext|>
            out.append("")
            continue
        out.append(raw.decode("utf-8", errors="replace"))
    return out


class TokenMask:
    def __init__(self, vocab, grammar, eos_id):
        self.vocab, self.grammar, self.eos_id = vocab, grammar, eos_id
        self._cache = {}

    def for_state(self, state):
        """Boolean mask over the vocabulary, plus the state each legal token lands in."""
        if state in self._cache:
            return self._cache[state]

        mask = torch.zeros(len(self.vocab), dtype=torch.bool)
        moves = {}
        if state == ACCEPT:
            mask[self.eos_id] = True
        else:
            for token_id, text in enumerate(self.vocab):
                if not text:
                    continue
                landed = self.grammar.consume(state, text)
                if landed is not None:
                    mask[token_id] = True
                    moves[token_id] = landed

        self._cache[state] = (mask, moves)
        return mask, moves


VOCAB = vocab_strings(tokenizer)
masker = TokenMask(VOCAB, grammar, tokenizer.eos_token_id)

mask, _ = masker.for_state(grammar.start())
print("legal at the very first step:", [VOCAB[i] for i in mask.nonzero().flatten()])

Two tokens out of 50,257. Both of them start the record correctly, and one of them
also commits the first quote of the first key.

## 4. One loop, four strategies

The decoding loop is the same for all four. Only the treatment of the logits
changes.

In [ ]:
def generate(prompt, constrained, temperature=0.0, max_new_tokens=40, seed=None,
             grammar=grammar, masker=masker):
    """Greedy or sampled decoding, with the grammar mask optionally applied."""
    if seed is not None:
        torch.manual_seed(seed)

    ids = tokenizer(prompt, return_tensors="pt").input_ids
    state, past, pieces = grammar.start(), None, []
    stats = {"allowed": [], "kept_mass": [], "steps": []}

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(ids, past_key_values=past, use_cache=True)
        past = out.past_key_values
        logits = out.logits[0, -1, :]

        if constrained:
            mask, moves = masker.for_state(state)
            probs = torch.softmax(logits.float(), dim=-1)
            stats["allowed"].append(int(mask.sum()))
            stats["kept_mass"].append(float(probs[mask].sum()))
            scores = logits.masked_fill(~mask, float("-inf"))
        else:
            scores = logits

        if temperature <= 0:
            token = int(scores.argmax())
        else:
            token = int(torch.multinomial(torch.softmax(scores / temperature, -1), 1))

        piece = tokenizer.decode([token])
        if token == tokenizer.eos_token_id or (not constrained and "\n" in piece):
            pieces.append(piece.split("\n")[0])
            break

        pieces.append(piece)
        stats["steps"].append(piece)
        ids = torch.tensor([[token]])
        if constrained:
            state = moves[token]
            if state == ACCEPT:
                break

    return "".join(pieces).strip(), stats


REQUIRED = {"name": str, "age": int, "verified": bool}


def parses(text):
    try:
        value = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return None
    return value if isinstance(value, dict) else None


def schema_ok(value):
    if value is None or set(value) != set(REQUIRED):
        return False
    return all(type(value[k]) is t for k, t in REQUIRED.items())


def fields_correct(value, gold):
    if not schema_ok(value):
        return False
    return (value["name"].strip().lower() == gold.name.lower()
            and value["age"] == gold.age and value["verified"] is gold.verified)


text, stats = generate(build_prompt(EVAL[0]), constrained=True)
print(text)
print("legal tokens per step:", stats["allowed"])

Now the comparison. `repair` is the pragmatic middle ground that most production
code implements: cut to the first brace, close what is open, and wrap a bare
fragment that looks like key-value pairs.

In [ ]:
_OBJECT = re.compile(r"\{.*", re.DOTALL)
_PAIR = re.compile(r'"[^"]+"\s*:')


def repair(text):
    """Cut to the first object and close what is open."""
    match = _OBJECT.search(text)
    if match is None:
        if _PAIR.search(text):
            return repair("{" + text.lstrip("[ "))
        return text

    out, depth, in_string, escaped = [], 0, False, False
    for ch in match.group(0):
        out.append(ch)
        if escaped:
            escaped = False
            continue
        if ch == "\\" and in_string:
            escaped = True
        elif ch == '"':
            in_string = not in_string
        elif not in_string:
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return "".join(out)
    if in_string:
        out.append('"')
    out.append("}" * max(depth, 0))
    return "".join(out).rstrip(", ")


def run(strategy, record, temperature, seed):
    prompt = build_prompt(record)
    if strategy == "constrained":
        text, _ = generate(prompt, constrained=True, temperature=temperature, seed=seed)
        return text, 1
    if strategy == "repair":
        text, _ = generate(prompt, constrained=False, temperature=temperature, seed=seed)
        return repair(text), 1
    if strategy == "ask":
        text, _ = generate(prompt, constrained=False, temperature=temperature, seed=seed)
        return text, 1
    for attempt in range(5):                    # retry until it parses
        text, _ = generate(prompt, constrained=False, temperature=temperature,
                           seed=seed + 1000 * attempt)
        if schema_ok(parses(text)):
            return text, attempt + 1
    return text, 5


for temperature in (0.0, 0.8):
    print(f"\n== {'greedy' if temperature == 0 else f'temperature {temperature}'} ==")
    for strategy in ("ask", "retry", "repair", "constrained"):
        results = [run(strategy, r, temperature, 7 * i) for i, r in enumerate(EVAL)]
        conforms = sum(schema_ok(parses(t)) for t, _ in results)
        correct = sum(fields_correct(parses(t), r) for (t, _), r in zip(results, EVAL))
        calls = sum(c for _, c in results) / len(results)
        print(f"  {strategy:<12} schema {conforms}/{len(EVAL)}"
              f"   correct {correct}/{len(EVAL)}   calls {calls:.2f}")

At temperature 0 every unconstrained strategy lands on the same number, because
they all start from the same deterministic sample. Retrying a temperature-0 call
returns the identical failure, which is worth remembering before wrapping one in a
retry loop.

Turn the temperature up and the prompted strategies collapse while the mask holds
at 8 of 8. The blog post runs this over 16 records and 3 seeds; the numbers there
are 4% for asking nicely, 23% for retrying, and 100% for the mask.

## 5. What the mask costs

### Sub-word misalignment

Re-encode a constrained output with GPT-2's own tokeniser and compare it to the
tokens the loop emitted.

In [ ]:
text, stats = generate(build_prompt(EVAL[0]), constrained=True)
canonical = [tokenizer.decode([i]) for i in tokenizer(text).input_ids]
print("emitted  :", stats["steps"])
print("canonical:", canonical)
print("same:", stats["steps"] == canonical)

The mask allows both `{` and `{"` at step one, the model scores the bare brace
higher, and the decode commits to a tokenisation GPT-2 would never have produced
for that string. Everything afterwards is conditioned on a prefix that is slightly
off distribution. Beurer-Kellner et al. (2024) named this and built DOMINO to keep
several candidate tokenisations alive instead of committing to the first legal one.

### The grammar is a hyperparameter

Three grammars, one schema. All three accept nothing but valid JSON. They do not
score the same.

In [ ]:
VARIANTS = {
    "canonical": dict(order=("name", "age", "verified"), space_after_colon=True),
    "reordered": dict(order=("verified", "name", "age"), space_after_colon=True),
    "compact": dict(order=("name", "age", "verified"), space_after_colon=False),
}

for label, kwargs in VARIANTS.items():
    g = Grammar(build_segments(**kwargs))
    m = TokenMask(VOCAB, g, tokenizer.eos_token_id)
    correct, mass = 0, []
    for i, record in enumerate(EVAL):
        text, stats = generate(build_prompt(record), constrained=True,
                               temperature=0.8, seed=7 * i, grammar=g, masker=m)
        correct += fields_correct(parses(text), record)
        mass += stats["kept_mass"]
    print(f"  {label:<10} correct {correct}/{len(EVAL)}"
          f"   mean probability mass kept {sum(mass) / len(mass):.3f}")

Deleting one space per field costs accuracy while leaving the guarantee untouched.
Write the grammar the way the model has seen the format written: match the spacing
in your examples, keep the key order stable, and re-evaluate after a schema change.

## 6. No download required

Everything above needs GPT-2's weights. The mechanism does not. Here is a scripted
stand-in that prefers one continuation (chatty prose wrapped around almost-JSON)
and falls back to a second (the correct record) when the first is unavailable.
Unconstrained it produces prose that does not parse; under the mask every prose
token is illegal, so the fallback is what survives.

In [ ]:
class StubTokenizer:
    """A tiny vocabulary: single characters plus a few merged pieces.

    The merged pieces are the point. `{"` and `":` cross grammar boundaries exactly
    as GPT-2's do, so the masking code faces the same problem at small scale.
    """

    MERGED = ['{"', '":', '", "', '": "', ", ", "true", "false", "JSON", "Sure"]

    def __init__(self):
        singles = [chr(c) for c in range(32, 127)] + ["\n"]
        self.vocab_texts = ["<|end|>"] + self.MERGED + singles
        self.eos_token_id = 0
        self._by_text = {t: i for i, t in enumerate(self.vocab_texts)}

    def __len__(self):
        return len(self.vocab_texts)

    def encode(self, text):
        ids, i = [], 0
        while i < len(text):
            for piece in sorted(self.MERGED, key=len, reverse=True):
                if text.startswith(piece, i):
                    ids.append(self._by_text[piece])
                    i += len(piece)
                    break
            else:
                ids.append(self._by_text.get(text[i], self._by_text[" "]))
                i += 1
        return ids

    def decode(self, ids):
        return "".join("" if i == self.eos_token_id else self.vocab_texts[i] for i in ids)


class ScriptedLM:
    def __init__(self, tokenizer):
        self.tokenizer, self.generated, self.prose, self.correct = tokenizer, "", "", ""

    def aim_at(self, record):
        self.prose = (f"Sure, here is the JSON: {{'name': '{record.name}', "
                      f"'age': {record.age}}}\n")
        self.correct = record.target

    def logits(self, generated):
        scores = torch.full((len(self.tokenizer),), -8.0)
        for token_id, text in enumerate(self.tokenizer.vocab_texts):
            if not text or token_id == self.tokenizer.eos_token_id:
                continue
            if self.prose.startswith(generated + text):
                scores[token_id] = 8.0 + len(text)
            elif self.correct.startswith(generated + text):
                scores[token_id] = 4.0 + len(text)
            else:
                scores[token_id] = -8.0 + len(text) * 0.01
        return scores


def stub_generate(stub, constrained, max_new_tokens=80):
    stub_grammar = Grammar()
    stub_mask = TokenMask(stub.tokenizer.vocab_texts, stub_grammar,
                          stub.tokenizer.eos_token_id)
    state, generated = stub_grammar.start(), ""
    for _ in range(max_new_tokens):
        scores = stub.logits(generated)
        if constrained:
            mask, moves = stub_mask.for_state(state)
            scores = scores.masked_fill(~mask, float("-inf"))
        token = int(scores.argmax())
        piece = stub.tokenizer.decode([token])
        if not piece or "\n" in piece:
            break
        generated += piece
        if constrained:
            state = moves[token]
            if state == ACCEPT:
                break
    return generated


stub = ScriptedLM(StubTokenizer())
stub.aim_at(EVAL[0])
loose = stub_generate(stub, constrained=False)
stub.aim_at(EVAL[0])
tight = stub_generate(stub, constrained=True)

print("unconstrained:", loose, "->", "parses" if parses(loose) else "does not parse")
print("constrained  :", tight, "->", "parses" if parses(tight) else "does not parse")

## Exercises

1. **Add an optional field.** Give the schema a `"note"` key that may be `null`.
   The automaton needs an alternation between a quoted string and the literal
   `null`, the same shape the boolean already uses.
2. **Measure the guarantee.** Run the constrained sampler at temperature 2.0 over
   all 8 records. Success should stay at 8 of 8 while accuracy falls, because
   the mask does not care how badly the model is behaving inside the allowed set.
3. **Fix the misalignment.** At step one, force the tokeniser's own choice `{"`
   instead of taking the argmax over legal tokens. Compare accuracy across the
   prompt set, and see whether one token of alignment is worth anything at this
   scale.
4. **Build the index up front.** `TokenMask` fills its cache lazily. Enumerate the
   reachable states instead and precompute every mask before decoding starts, which
   is what Willard and Louf (2023) describe. Time both.
5. **Break the grammar deliberately.** Set `MAX_AGE_DIGITS = 1` and run over the
   whole set. Every output still parses; every age is now wrong. Constrained
   decoding guarantees the shape and nothing else.

## Further reading

- Geng et al. (2023), [Grammar-Constrained Decoding for Structured NLP Tasks without Finetuning](https://arxiv.org/abs/2305.13971)
- Willard and Louf (2023), [Efficient Guided Generation for Large Language Models](https://arxiv.org/abs/2307.09702)
- Beurer-Kellner et al. (2024), [Guiding LLMs The Right Way](https://arxiv.org/abs/2403.06988)
- [Outlines](https://github.com/dottxt-ai/outlines), the reference implementation